# Calcualting aggregate statistic values

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

In [2]:
DATA_PATH = 'reports'
ALL_DATA = os.path.join(DATA_PATH, 'all.parquet')
COEF_VAR = 'b'

In [3]:
df = pd.read_parquet(ALL_DATA)
df.shape

(7929, 8)

In [4]:
df.head()

,corpus,lang,cid,speaker,self,a,b,k
0,callhome,deu,xling-callhome-deu-callhome-deu-6692,callhome-deu-callhome-deu-6692-A,True,0.000013,-0.379041,13931
1,callhome,deu,xling-callhome-deu-callhome-deu-6692,callhome-deu-callhome-deu-6692-A,False,0.000011,-0.273302,12738
2,callhome,deu,xling-callhome-deu-callhome-deu-6692,callhome-deu-callhome-deu-6692-B,False,0.000017,-0.468889,12117
3,callhome,deu,xling-callhome-deu-callhome-deu-6692,callhome-deu-callhome-deu-6692-B,True,0.000025,-0.495063,10914
4,finchat,fin,xling-finchat-fin-finchat-fin-63,finchat-fin-63-54,True,0.000434,0.437574,36


## Merging data

In [5]:
df = pd.merge(
    left=df.loc[df['self']], left_on=['cid', 'speaker'],
    right=df[['cid', 'speaker', COEF_VAR, 'a']].loc[~df['self']], right_on=['cid', 'speaker'],
    how='inner'
)
df.shape

(4780, 10)

In [6]:
df.isna().sum()

corpus     0
lang       0
cid        0
speaker    0
self       0
a_x        0
b_x        0
k          0
b_y        0
a_y        0
dtype: int64

In [7]:
# df = df.loc[~df['b_y'].isna()]

In [8]:
df['b_delta'] = df['b_y'] - df['b_x']
# df['b_delta'] = np.log(np.abs( np.exp(df['b_x']) - np.exp(df['b_y']) ))
# df['b_delta'] = np.log(
#     np.abs(
#         np.exp(df['b_x'] + df['a_x']) - np.exp(df['b_y'] + df['a_y']) + 1e-9
#     )
# )


## T-statistics

### Difference from zero

In [9]:
null = 0

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values

dof = len(vs)-1

tstat = (vs.mean() - null) / (vs.std() / np.sqrt(len(vs)-1))

# tstat = ((df0[COEF_VAR] - null) / df0['se']).values
vs.mean(), vs.std(), dof, tstat, t.sf(tstat.__abs__(), df=dof)

(np.float64(0.06906529690173056),
 np.float64(0.44840468422329705),
 4779,
 np.float64(10.647760098376855),
 np.float64(1.751602830814488e-26))

### Difference from identity

In [10]:
null = 1

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values

tstat = (vs.mean() - null) / (vs.std() / np.sqrt(len(vs)-1))

# tstat = ((df0[COEF_VAR] - null) / df0['se']).values
vs.mean(), vs.std(), tstat, t.sf(tstat.__abs__(), df=len(df) - 1)

(np.float64(0.06906529690173056),
 np.float64(0.44840468422329705),
 np.float64(-143.52170815898836),
 np.float64(0.0))

### Wilcoxan Signed Rank Test

Because I'm not sure I want to bet on the data being normally distributed...

In [11]:
from scipy.stats import wilcoxon

In [12]:
null = 0

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values - null

wilcoxon(vs)

WilcoxonResult(statistic=np.float64(3772361.0), pvalue=np.float64(5.477261426044847e-92))

In [13]:
null = 1

vs = df['b_delta'].loc[
    df['self']
    & (~np.isinf(df['b_delta']))
    & (~df['b_delta'].isna())
].values - null

wilcoxon(vs)

WilcoxonResult(statistic=np.float64(322850.0), pvalue=np.float64(0.0))

## Visualizations!

In [13]:
import plotly.figure_factory as ff

In [14]:
hist_data = [
    # df['b_delta'].loc[
    #     (~df['self'])
    #     & (~np.isinf(df['b_delta']))
    #     & (~df['b_delta'].isna())
    # ].values,
    df['b_delta'].loc[
        df['self']
        & (~np.isinf(df['b_delta']))
        & (~df['b_delta'].isna())
    ].values,
]
# print(len(hist_data[0]),len(hist_data[1]),)

groups = [
    # 'other',
    'self',
]

colors = [
    # '#0504aa',
    '#FFC300',
]

In [15]:
fig = ff.create_distplot(hist_data,groups,show_hist=False,colors=colors)
fig.update_layout(template='ygridoff',showlegend=False)

In [16]:
fig.write_html(os.path.join(DATA_PATH, 'CM-results.html'))